In [1]:
import pandas as pd
import numpy as np
import torch
from pytorch_forecasting.models import TemporalFusionTransformer
from pytorch_forecasting.data import TimeSeriesDataSet
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

PATH_H1 = r"C:\Users\admin\Desktop\Forex_algo\code\tft_ret_rich_v5.ckpt"
PATH_H4 = r"c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_57\checkpoints\eurusd_h4_tft-epoch=20-val_loss=0.001522.ckpt"
DATA_H1 = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet"
DATA_H4 = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H4.parquet"


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:28: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
tft_h1 = TemporalFusionTransformer.load_from_checkpoint(PATH_H1)
tft_h4 = TemporalFusionTransformer.load_from_checkpoint(PATH_H4)

print("H1 model loaded!")
print("H4 model loaded!")


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


H1 model loaded!
H4 model loaded!


In [3]:
df_h1 = pd.read_parquet(DATA_H1)
df_h4 = pd.read_parquet(DATA_H4)

print("H1:", df_h1.shape)
print("H4:", df_h4.shape)


H1: (61473, 14)
H4: (15375, 14)


In [4]:
def prepare_df(df):
    df = df.copy()

    # Ensure time column exists
    if "time" not in df.columns:
        raise ValueError("Dataframe has no 'time' column. Check the parquet file.")

    # Convert time column to datetime
    df["time"] = pd.to_datetime(df["time"], utc=True)

    # Set datetime index
    df = df.set_index("time")

    # Add required columns
    df["time_idx"] = np.arange(len(df))
    df["series_id"] = 0
    df["hour"] = df.index.hour
    df["day_of_week"] = df.index.dayofweek

    return df

# Apply to both
h1_eval = prepare_df(df_h1)
h4_eval = prepare_df(df_h4)

h1_eval.head(), h4_eval.head()


(                           volume    mid_o    mid_h    mid_l    mid_c  \
 time                                                                    
 2016-01-07 00:00:00+00:00     542  1.07764  1.07832  1.07744  1.07778   
 2016-01-07 01:00:00+00:00    3167  1.07776  1.08100  1.07748  1.08029   
 2016-01-07 02:00:00+00:00    1567  1.08026  1.08176  1.07996  1.08152   
 2016-01-07 03:00:00+00:00     914  1.08156  1.08257  1.08150  1.08187   
 2016-01-07 04:00:00+00:00     649  1.08190  1.08256  1.08156  1.08236   
 
                              bid_o    bid_h    bid_l    bid_c    ask_o  \
 time                                                                     
 2016-01-07 00:00:00+00:00  1.07757  1.07823  1.07735  1.07770  1.07772   
 2016-01-07 01:00:00+00:00  1.07768  1.08092  1.07740  1.08020  1.07784   
 2016-01-07 02:00:00+00:00  1.08018  1.08169  1.07987  1.08144  1.08035   
 2016-01-07 03:00:00+00:00  1.08147  1.08249  1.08142  1.08178  1.08164   
 2016-01-07 04:00:00+00:00  1.

In [5]:
import numpy as np
from pathlib import Path
import sys

# Make sure your project root is on sys.path so we can import indicators.py
project_root = Path(r"C:\Users\admin\Desktop\Forex_algo\code")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from technicals.indicators import BollingerBands, ATR, KeltnerChannels, RSI, MACD


def add_standard_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- Core indicators from your file ---
    df = BollingerBands(df)        # BB_MA, BB_UP, BB_LW
    df = ATR(df, n=14)             # ATR_14
    df = RSI(df, n=14)             # RSI_14
    df = MACD(df)                  # MACD, SIGNAL, HIST

    # --- EMAs (5 / 20 / 50 / 200) on mid_c ---
    for span in (5, 20, 50, 200):
        df[f"EMA_{span}"] = df["mid_c"].ewm(span=span, min_periods=span).mean()

    # --- Aliases in lowercase for the TFT feature list ---
    df["rsi"]          = df["RSI_14"]
    df["macd"]         = df["MACD"]
    df["macd_signal"]  = df["SIGNAL"]
    df["bb_lower"]     = df["BB_LW"]
    df["bb_middle"]    = df["BB_MA"]
    df["bb_upper"]     = df["BB_UP"]
    df["atr14"]        = df["ATR_14"]
    df["ema_5"]        = df["EMA_5"]
    df["ema_20"]       = df["EMA_20"]
    df["ema_50"]       = df["EMA_50"]
    df["ema_200"]      = df["EMA_200"]

    # --- Extra engineered features (used in your rich model doc) ---
    df["momentum_oc"]        = df["mid_o"] - df["mid_c"]
    df["avg_price_hl"]       = (df["mid_h"] + df["mid_l"]) / 2
    df["range_hl"]           = df["mid_h"] - df["mid_l"]
    df["typical_price_ohlc"] = (df["mid_o"] + df["mid_h"] + df["mid_l"] + df["mid_c"]) / 4
    df["log_volume"]         = np.log(df["volume"].replace(0, np.nan))

    return df


In [6]:
# 1) Add indicators
h1_eval = add_standard_indicators(h1_eval)
h4_eval = add_standard_indicators(h4_eval)

# 2) Add targets (next candle log-return)
h1_eval["target_return"] = np.log(h1_eval["mid_c"].shift(-1)) - np.log(h1_eval["mid_c"])
h4_eval["target_return_4h"] = np.log(h4_eval["mid_c"].shift(-1)) - np.log(h4_eval["mid_c"])

# 3) Drop NaNs (from indicators and last target row)
h1_eval = h1_eval.dropna().copy()
h4_eval = h4_eval.dropna().copy()

# 4) Rebuild time_idx after dropna so it stays contiguous
h1_eval["time_idx"] = np.arange(len(h1_eval))
h4_eval["time_idx"] = np.arange(len(h4_eval))

# 5) Make hour/day_of_week categorical (required by TimeSeriesDataSet)
for df in [h1_eval, h4_eval]:
    df["hour"] = df["hour"].astype(str)
    df["day_of_week"] = df["day_of_week"].astype(str)

h1_eval.head(), h4_eval.head()


(Empty DataFrame
 Columns: [volume, mid_o, mid_h, mid_l, mid_c, bid_o, bid_h, bid_l, bid_c, ask_o, ask_h, ask_l, ask_c, time_idx, series_id, hour, day_of_week, BB_MA, BB_UP, BB_LW, ATR_14, RSI_14, MACD, SIGNAL, HIST, EMA_5, EMA_20, EMA_50, EMA_200, rsi, macd, macd_signal, bb_lower, bb_middle, bb_upper, atr14, ema_5, ema_20, ema_50, ema_200, momentum_oc, avg_price_hl, range_hl, typical_price_ohlc, log_volume, target_return]
 Index: [],
 Empty DataFrame
 Columns: [volume, mid_o, mid_h, mid_l, mid_c, bid_o, bid_h, bid_l, bid_c, ask_o, ask_h, ask_l, ask_c, time_idx, series_id, hour, day_of_week, BB_MA, BB_UP, BB_LW, ATR_14, RSI_14, MACD, SIGNAL, HIST, EMA_5, EMA_20, EMA_50, EMA_200, rsi, macd, macd_signal, bb_lower, bb_middle, bb_upper, atr14, ema_5, ema_20, ema_50, ema_200, momentum_oc, avg_price_hl, range_hl, typical_price_ohlc, log_volume, target_return_4h]
 Index: [])

In [9]:
# Make categorical columns actually categorical (string or category)
for df in [h1_eval, h4_eval]:
    df["hour"] = df["hour"].astype(str)         # or .astype("category")
    df["day_of_week"] = df["day_of_week"].astype(str)


In [10]:
FEATURE_COLS = [
    "mid_o","mid_h","mid_l","mid_c",
    "rsi","macd","macd_signal","atr14",
    "ema_5","ema_20","ema_50","ema_200",
    "bb_lower","bb_middle","bb_upper"
]

In [12]:
from pytorch_forecasting.data import TimeSeriesDataSet
from torch.utils.data import DataLoader

# H1 dataset
test_h1 = TimeSeriesDataSet(
    h1_eval,
    time_idx="time_idx",
    target="target_return",
    group_ids=["series_id"],
    max_encoder_length=96,
    max_prediction_length=1,
    time_varying_unknown_reals=FEATURE_COLS,
    time_varying_known_categoricals=["hour", "day_of_week"],
)

test_loader_h1 = DataLoader(test_h1, batch_size=128, shuffle=False)

# H4 dataset
test_h4 = TimeSeriesDataSet(
    h4_eval,
    time_idx="time_idx",
    target="target_return_4h",
    group_ids=["series_id"],
    max_encoder_length=96,
    max_prediction_length=1,
    time_varying_unknown_reals=FEATURE_COLS,
    time_varying_known_categoricals=["hour", "day_of_week"],
)

test_loader_h4 = DataLoader(test_h4, batch_size=128, shuffle=False)


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\numpy\core\_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\numpy\core\_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\numpy\core\_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by StandardScaler.